In [ ]:
!pip install sentence-transformers rank-bm25 transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 8.8 MB/s eta 0:00:00


In [ ]:
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
corpus = [
    "Transformers use self-attention mechanisms to process sequences in parallel.",
    "Attention allows models to focus on relevant parts of the input sequence.",
    "Gradient descent is used to optimize neural network parameters.",
    "Backpropagation computes gradients for training deep neural networks.",
    "Adam optimizer combines momentum and adaptive learning rates.",
    "Convolutional neural networks are used for image processing tasks.",
    "Recurrent neural networks process sequential data step by step.",
    "BERT is a transformer-based model trained using masked language modeling.",
    "Dropout is a regularization technique to prevent overfitting.",
    "XGBoost is a gradient boosting algorithm often used in structured data tasks."
]

In [ ]:
class HybridRetriever:
    def __init__(self, corpus, k=60):
        self.corpus = corpus
        self.k = k

        # BM25 setup
        self.tokenized_corpus = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(self.tokenized_corpus)

        # SBERT setup
        self.sbert = SentenceTransformer('all-MiniLM-L6-v2')
        self.doc_embeddings = self.sbert.encode(corpus)

    def retrieve(self, query, top_k=5):
        # BM25
        tokenized_query = query.lower().split()
        bm25_scores = self.bm25.get_scores(tokenized_query)
        bm25_ranks = np.argsort(bm25_scores)[::-1]

        # SBERT
        query_embedding = self.sbert.encode([query])
        sim_scores = cosine_similarity(query_embedding, self.doc_embeddings)[0]
        sbert_ranks = np.argsort(sim_scores)[::-1]

        # RRF Fusion
        results = []
        for i in range(len(self.corpus)):
            bm25_rank = np.where(bm25_ranks == i)[0][0] + 1
            sbert_rank = np.where(sbert_ranks == i)[0][0] + 1

            rrf_score = (1 / (self.k + bm25_rank)) + (1 / (self.k + sbert_rank))

            results.append({
                "doc_id": i,
                "rrf_score": rrf_score,
                "bm25_rank": bm25_rank,
                "sbert_rank": sbert_rank,
                "text": self.corpus[i]
            })

        results = sorted(results, key=lambda x: x["rrf_score"], reverse=True)
        return results[:top_k]

In [ ]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def rerank(query, candidates, top_k=3):
    pairs = [(query, doc["text"]) for doc in candidates]
    scores = cross_encoder.predict(pairs)

    for i, doc in enumerate(candidates):
        doc["cross_score"] = scores[i]

    reranked = sorted(candidates, key=lambda x: x["cross_score"], reverse=True)
    return reranked[:top_k]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [ ]:
def expand_query(query):
    expansions = [
        query,
        f"Explain {query}",
        f"What is {query} in machine learning?"
    ]
    return expansions

In [ ]:
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = sbert_model.encode(corpus)

def naive_rag(query):
    q_emb = sbert_model.encode([query])
    scores = cosine_similarity(q_emb, doc_embeddings)[0]
    top_idx = np.argmax(scores)
    return corpus[top_idx]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
retriever = HybridRetriever(corpus)

def advanced_rag(user_query):
    # Step 1: Query Expansion
    queries = expand_query(user_query)

    # Step 2: Retrieve for each query
    all_results = []
    for q in queries:
        all_results.extend(retriever.retrieve(q, top_k=5))

    # Deduplicate
    unique = {}
    for doc in all_results:
        unique[doc["text"]] = doc

    candidates = list(unique.values())

    # Step 3: Re-rank
    top_docs = rerank(user_query, candidates, top_k=3)

    # Step 4: Generate answer (simple join instead of LLM)
    context = " ".join([doc["text"] for doc in top_docs])

    answer = f"Answer based on retrieved context:\n{context}"
    return answer, top_docs

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "what is dropout in neural networks?"
]

print("=== COMPARISON ===\n")

for q in queries:
    naive = naive_rag(q)
    adv_answer, adv_docs = advanced_rag(q)
    adv_top = adv_docs[0]["text"]

    print(f"Query: {q}")
    print(f"Naive Top Doc: {naive}")
    print(f"Advanced Top Doc: {adv_top}")
    print(f"Different?: {naive != adv_top}")
    print("-" * 60)

=== COMPARISON ===

Query: how do transformers encode meaning?
Naive Top Doc: Transformers use self-attention mechanisms to process sequences in parallel.
Advanced Top Doc: Transformers use self-attention mechanisms to process sequences in parallel.
Different?: False
------------------------------------------------------------
Query: optimization techniques for training
Naive Top Doc: Gradient descent is used to optimize neural network parameters.
Advanced Top Doc: Backpropagation computes gradients for training deep neural networks.
Different?: True
------------------------------------------------------------
Query: what is dropout in neural networks?
Naive Top Doc: Dropout is a regularization technique to prevent overfitting.
Advanced Top Doc: Dropout is a regularization technique to prevent overfitting.
Different?: False
------------------------------------------------------------


| Query | Naïve RAG Top Doc | Advanced RAG Top Doc | Are they different? |
|---|---|---|---|
| how do transformers encode meaning? | Transformers use self-attention mechanisms to process sequences in parallel. | Transformers use self-attention mechanisms to process sequences in parallel. | No |
| optimization techniques for training | Gradient descent is used to optimize neural network parameters. | Backpropagation computes gradients for training deep neural networks. | Yes |
| what is dropout in neural networks? | Dropout is a regularization technique to prevent overfitting. | Dropout is a regularization technique to prevent overfitting. | No |